# v20 — Regime-Switching Multi-Strategy (Trending vs Ranging)

**Latar belakang:** Pengamatan langsung dari chart live (17 Agustus 2026) & trade log live v13
menunjukkan pola yang konsisten dgn temuan v16/v19 sebelumnya -- v13 (trend-following +
momentum exhaustion) itu **cocok saat trend kuat**, tapi rawan LOSS saat market masuk fase
**choppy/sideways setelah trend memuncak** (candle bolak-balik di range sempit, momentum chain
6-7 -- "hampir exhausted" tapi belum dapat perlakuan SL/TP diperkecil spt chain 8).

Trade live `92286674` (17 Agustus, entry 04:00 UTC BUY di ADX=39.8, bull_chain=7, LOSS -$9.87)
jadi bukti konkret: robot entry di puncak momentum, harga langsung berbalik & kena SL persis di
fase harga jadi ranging/choppy sesudahnya.

**Ide besar user**: drpd 1 strategi (v13) dipaksa jalan di semua kondisi, bangun **robot yang
mendeteksi regime market dulu** (trending kuat / ranging tenang / ranging choppy / gak jelas),
lalu **pilih strategi yang sesuai**:
- **Trending kuat** (naik atau turun) -> Strategi A = **v13 apa adanya** (sudah tervalidasi cocok
  di regime ini, lihat v16: Trending+High Vol = regime terkuat, PF 3.19-4.22)
- **Ranging tenang** -> Strategi B = **mean-reversion** (kebalikan logika v13 -- beli di area
  oversold/bawah range, jual di overbought/atas range), BELUM ADA, perlu dieksplorasi dari nol
- **Ranging choppy/whipsaw** -> Strategi C = **skip/tidak trading** (konsisten dgn temuan v18/v19:
  kondisi ini yang paling berisiko & paling sulit ditemukan edge-nya)

**Cakupan v20 ini**: (1) definisikan klasifikasi regime 4-kategori yang lebih tajam dari v16
(yang cuma pakai split biner ADX/ATR median), pakai ADX (kekuatan) + slope arah (arah) + ukuran
choppy (frekuensi whipsaw); (2) ukur ulang performa v13 MURNI per kategori regime baru ini
(extension v16, definisi lebih tajam); (3) eksplorasi (BEST EFFORT, bukan janji berhasil) apakah
ada logika mean-reversion yang profitable khusus di kategori Ranging-Tenang.

**Kriteria kejujuran (sama spt v19)**: kalau strategi B (mean-reversion) tidak ketemu kombinasi
yang PF>1.5 di TEST out-of-sample dgn sample cukup (>=30 TRAIN, >=15 TEST), kesimpulannya JUJUR
"ranging-tenang juga belum ada edge yang ditemukan" -- bukan dipaksakan. TIDAK ADA perubahan ke
`usecase.py` dari notebook ini -- murni riset.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v20"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / VERSION).mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82  # spread broker MIFX riil, dari v18

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load data 2025-2026 (rezim volatilitas tinggi, era v13) + definisikan regime 4-kategori

Beda dari v16 (split biner ADX>=25 Trending vs <25 Ranging, ATR high/low split median), di sini
regime dipecah lebih tajam jadi 4 kategori pakai **ADX (kekuatan trend) + `close_slope` (arah)
+ ukuran choppy (whipsaw)**:

- **Trending-Naik**: ADX >= 25 DAN `close_slope` (rata-rata beberapa candle) positif jelas
- **Trending-Turun**: ADX >= 25 DAN `close_slope` negatif jelas
- **Ranging-Tenang**: ADX < 25 DAN whipsaw_score RENDAH (arah candle konsisten dlm range sempit,
  bukan bolak-balik liar)
- **Ranging-Choppy**: ADX < 25 DAN whipsaw_score TINGGI (arah candle sering ganti tanda -- ini
  kondisi paling berbahaya, konsisten dgn temuan v18/v19)

**whipsaw_score** = proporsi candle dlm window N=12 (1 jam) yang arahnya (naik/turun) BEDA dari
candle sebelumnya -- makin tinggi, makin sering bolak-balik (choppy).

In [2]:
REGIME_CACHE_PATH = PROCESSED_DIR / VERSION / "df_2025_2026_regime4.parquet"

if REGIME_CACHE_PATH.exists():
    print(f"Load dari cache: {REGIME_CACHE_PATH}")
    df = pd.read_parquet(REGIME_CACHE_PATH)
else:
    print("Belum ada cache -- load & proses dari awal...")
    df = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_m5_full_indicators.csv")
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    df = df[df["datetime"] >= pd.Timestamp("2025-01-01", tz="UTC")].reset_index(drop=True)

    # Arah candle (naik=1/turun=-1/flat=0), lalu whipsaw = proporsi arah BEDA dari candle sblmnya
    # dlm window 12 candle (1 jam M5) -- makin tinggi, makin sering ganti arah (choppy).
    candle_dir = np.sign(df["close"].diff())
    dir_change = (candle_dir != candle_dir.shift(1)).astype(float)
    df["whipsaw_score"] = dir_change.rolling(12, min_periods=6).mean()

    # slope arah trend -- rata-rata close_slope beberapa candle terakhir (lebih stabil dari 1 titik)
    df["slope_smooth"] = df["close_slope"].rolling(6, min_periods=3).mean()

    df.to_parquet(REGIME_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {REGIME_CACHE_PATH}")

print(f"Total candle: {len(df)}, {df['datetime'].min()} -> {df['datetime'].max()}")

Belum ada cache -- load & proses dari awal...


Tersimpan ke cache: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v20\df_2025_2026_regime4.parquet
Total candle: 107336, 2025-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00


In [3]:
ADX_TRENDING_MIN = 25.0
WHIPSAW_HIGH_THRESHOLD = df["whipsaw_score"].median()  # split median, adil lintas periode

def classify_regime4(row) -> str:
    adx = row["adx"]
    slope = row["slope_smooth"]
    whipsaw = row["whipsaw_score"]
    if pd.isna(adx) or pd.isna(slope) or pd.isna(whipsaw):
        return "UNKNOWN"
    if adx >= ADX_TRENDING_MIN:
        return "Trending-Naik" if slope > 0 else "Trending-Turun"
    return "Ranging-Choppy" if whipsaw >= WHIPSAW_HIGH_THRESHOLD else "Ranging-Tenang"

df["regime4"] = df.apply(classify_regime4, axis=1)

print(f"whipsaw_score median (threshold choppy): {WHIPSAW_HIGH_THRESHOLD:.3f}")
print()
print("=== Distribusi regime4 (semua candle 2025-2026) ===")
print(df["regime4"].value_counts())
print()
print("=== Proporsi (%) ===")
print((df["regime4"].value_counts(normalize=True) * 100).round(1))

whipsaw_score median (threshold choppy): 0.500

=== Distribusi regime4 (semua candle 2025-2026) ===
regime4
Ranging-Choppy    41208
Trending-Naik     23132
Trending-Turun    21924
Ranging-Tenang    21067
UNKNOWN               5
Name: count, dtype: int64

=== Proporsi (%) ===
regime4
Ranging-Choppy    38.4
Trending-Naik     21.6
Trending-Turun    20.4
Ranging-Tenang    19.6
UNKNOWN            0.0
Name: proportion, dtype: float64


## 2. Baseline -- performa v13 MURNI per kategori regime4

Reuse trade log v13 final (`v13/trade_log_full_final.csv`, 721 trade, sudah lolos SEMUA filter
v13 termasuk Order Block & exhaustion handling) -- gabungkan dgn label regime4 di atas (join by
`entry_time`) supaya tau: **di kategori regime mana v13 kuat, di kategori mana lemah**, dgn
definisi regime yang lebih tajam dari v16 (yang cuma biner Trending/Ranging x High/Low Vol).

In [4]:
trades_v13 = pd.read_csv(PROCESSED_DIR / "v13" / "trade_log_full_final.csv")
trades_v13["entry_time"] = pd.to_datetime(trades_v13["entry_time"])

# Join ke regime4 by nearest candle (entry_time trade v13 harusnya persis cocok candle df)
regime_lookup = df[["datetime", "regime4"]].rename(columns={"datetime": "entry_time"})
trades_v13 = pd.merge_asof(
    trades_v13.sort_values("entry_time"), regime_lookup.sort_values("entry_time"),
    on="entry_time", direction="nearest", tolerance=pd.Timedelta("5min"),
)
print(f"Trade v13 berhasil di-label regime: {trades_v13['regime4'].notna().sum()} / {len(trades_v13)}")

def regime_performance(trades: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for regime, g in trades.groupby(group_col, observed=True):
        wins = (g["result"] == "WIN").sum()
        gross_profit = g.loc[g["pnl"] > 0, "pnl"].sum()
        gross_loss = g.loc[g["pnl"] <= 0, "pnl"].sum()
        rows.append({
            "regime": regime, "n_trades": len(g),
            "win_rate_pct": round(wins / len(g) * 100, 1) if len(g) else 0,
            "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
            "avg_pnl": round(g["pnl"].mean(), 2),
            "total_pnl": round(g["pnl"].sum(), 2),
        })
    return pd.DataFrame(rows).sort_values("profit_factor", ascending=False)

print()
print("=== Performa v13 MURNI per regime4 (721 trade, 2025-2026) ===")
print(regime_performance(trades_v13, "regime4").to_string(index=False))

Trade v13 berhasil di-label regime: 721 / 721

=== Performa v13 MURNI per regime4 (721 trade, 2025-2026) ===
        regime  n_trades  win_rate_pct  profit_factor  avg_pnl  total_pnl
Ranging-Tenang       161          77.6           7.32     8.83    1422.09
Trending-Turun       153          74.5           4.54    11.49    1758.19
 Trending-Naik       250          67.2           2.96     5.05    1263.70
Ranging-Choppy       157          65.0           2.24     4.47     701.29


## 3. Berapa banyak candle Ranging-Tenang yang TIDAK menghasilkan trade v13 sama sekali?

Section 2 di atas cuma lihat trade yang SUDAH lolos filter v13 (termasuk `adx_min=18`, jadi
beberapa Ranging-Tenang bisa saja ADX 18-25, masih lolos filter dasar). Tapi ADX<18 (mayoritas
candle ranging sesungguhnya) otomatis di-skip v13 -- ini "peluang hilang" yang jadi alasan
kenapa strategi B (mean-reversion) worth dieksplorasi: seberapa besar porsi waktu robot DIAM
krn regime ranging, dan apakah itu porsi yang signifikan dari total waktu trading.

In [5]:
print("=== Distribusi regime4 TERPISAH: candle dgn ADX < 18 (pasti di-skip v13 filter dasar) ===")
below_filter = df[df["adx"] < 18.0]
print(f"Total candle ADX<18: {len(below_filter)} ({len(below_filter)/len(df)*100:.1f}% dari semua candle)")
print(below_filter["regime4"].value_counts())
print()
print("Catatan: ADX<18 pasti Ranging-Tenang/Choppy (krn threshold Trending=25), ")
print("ini murni cek proporsi -- makin besar porsinya, makin besar potensi upside strategi B kalau berhasil.")

=== Distribusi regime4 TERPISAH: candle dgn ADX < 18 (pasti di-skip v13 filter dasar) ===
Total candle ADX<18: 30350 (28.3% dari semua candle)
regime4
Ranging-Choppy    20932
Ranging-Tenang     9418
Name: count, dtype: int64

Catatan: ADX<18 pasti Ranging-Tenang/Choppy (krn threshold Trending=25), 
ini murni cek proporsi -- makin besar porsinya, makin besar potensi upside strategi B kalau berhasil.


## 4. Eksplorasi Strategi B: mean-reversion khusus Ranging-Tenang (BEST EFFORT)

**Logika kebalikan dari v13**: alih-alih ikut momentum (beli saat kuat naik), mean-reversion
beli saat harga di area OVERSOLD relatif ke range-nya sendiri (dekat bawah, RSI rendah, BB %B
rendah), jual saat OVERBOUGHT (dekat atas, RSI tinggi, BB %B tinggi) -- profit dari harga
"mantul" balik ke tengah range, BUKAN dari breakout searah.

**Kandidat sinyal** (dicoba, bukan janji semuanya bekerja):
1. **BB %B ekstrem**: `bb_pct <= 0.1` (dekat lower band) -> BUY, `bb_pct >= 0.9` (dekat upper
   band) -> SELL
2. **RSI ekstrem**: `rsi <= 30` -> BUY, `rsi >= 70` -> SELL
3. **Kombinasi (BB %B DAN RSI sepakat)**: sinyal lebih ketat, cuma entry kalau KEDUA indikator
   sepakat ekstrem

**SL/TP mean-reversion beda logika dari v13**: TP di sekitar titik tengah range (`bb_mid`, bukan
kelipatan ATR searah momentum), SL di luar band (breakout gagal, harga TIDAK balik ke tengah).

In [6]:
def run_backtest_mean_reversion(
    df_signals: pd.DataFrame,
    signal_mode: str,  # "bb", "rsi", "both"
    bb_entry_pct: float,
    rsi_entry: float,
    sl_atr_mult: float,
    max_hold: int,
    spread_points: float = REAL_SPREAD,
) -> pd.DataFrame:
    """Backtest mean-reversion khusus candle Ranging-Tenang. TP = bb_mid (titik tengah range),
    SL = sl_atr_mult x ATR di luar entry (breakout gagal, mean-reversion TIDAK terjadi)."""
    sub = df_signals[df_signals["regime4"] == "Ranging-Tenang"].reset_index(drop=True)

    close_arr = sub["close"].to_numpy()
    high_arr = sub["high"].to_numpy()
    low_arr = sub["low"].to_numpy()
    atr_arr = sub["atr"].to_numpy()
    bb_pct_arr = sub["bb_pct"].to_numpy()
    bb_mid_arr = sub["bb_mid"].to_numpy()
    rsi_arr = sub["rsi"].to_numpy()
    datetime_arr = sub["datetime"].to_numpy()
    n = len(sub)

    trades = []
    equity = INITIAL_EQUITY
    i = 0
    while i < n:
        close, atr, bb_pct, bb_mid, rsi = close_arr[i], atr_arr[i], bb_pct_arr[i], bb_mid_arr[i], rsi_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(bb_pct) or not np.isfinite(rsi):
            i += 1
            continue

        buy_signal = sell_signal = False
        if signal_mode == "bb":
            buy_signal = bb_pct <= bb_entry_pct
            sell_signal = bb_pct >= (1 - bb_entry_pct)
        elif signal_mode == "rsi":
            buy_signal = rsi <= rsi_entry
            sell_signal = rsi >= (100 - rsi_entry)
        else:  # both
            buy_signal = bb_pct <= bb_entry_pct and rsi <= rsi_entry
            sell_signal = bb_pct >= (1 - bb_entry_pct) and rsi >= (100 - rsi_entry)

        if not (buy_signal or sell_signal):
            i += 1
            continue
        direction = "BUY" if buy_signal else "SELL"

        sl_points = sl_atr_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = bb_mid  # target = kembali ke tengah range
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        # Validasi TP masuk akal (BUY: TP harus di atas entry; SELL: TP di bawah entry)
        if (direction == "BUY" and tp_price <= entry_price) or (direction == "SELL" and tp_price >= entry_price):
            i += 1
            continue

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        risk_amount = equity * RISK_PCT
        lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

print("Backtest engine mean-reversion siap.")

Backtest engine mean-reversion siap.


## 5. TRAIN/TEST split & grid search strategi B

In [7]:
TRAIN_END = pd.Timestamp("2026-03-01", tz="UTC")  # konsisten dgn TRAIN/TEST v13 asli

df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN (s/d 2026-03-01): {len(df_train)} candle, Ranging-Tenang: {(df_train['regime4']=='Ranging-Tenang').sum()}")
print(f"TEST (2026-03-01 dst): {len(df_test)} candle, Ranging-Tenang: {(df_test['regime4']=='Ranging-Tenang').sum()}")

TRAIN (s/d 2026-03-01): 77226 candle, Ranging-Tenang: 15628
TEST (2026-03-01 dst): 30110 candle, Ranging-Tenang: 5439


In [8]:
import itertools
import time as _time

GRID = {
    "signal_mode": ["bb", "rsi", "both"],
    "bb_entry_pct": [0.05, 0.1, 0.15],
    "rsi_entry": [20.0, 25.0, 30.0],
    "sl_atr_mult": [1.0, 1.5, 2.0],
    "max_hold": [12],
}

combos = list(itertools.product(*GRID.values()))
print(f"Total kombinasi grid: {len(combos)}")

t0 = _time.time()
grid_results = []
for idx, combo in enumerate(combos):
    params = dict(zip(GRID.keys(), combo))
    trades = run_backtest_mean_reversion(df_train, **params)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics.update(params)
    grid_results.append(metrics)
    if (idx + 1) % 10 == 0:
        print(f"  [{idx+1}/{len(combos)}] {_time.time()-t0:.0f}s")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search selesai dalam {_time.time()-t0:.0f}s")

grid_valid = grid_df[grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 10 kandidat dgn sample TRAIN >= {MIN_SAMPLE_TRAIN} trade ===")
print(grid_valid.head(10).to_string(index=False))
print(f"\nTotal kandidat yang lolos sample minimum: {len(grid_valid)} dari {len(grid_df)}")

Total kombinasi grid: 81


  [10/81] 1s


  [20/81] 1s


  [30/81] 2s


  [40/81] 2s


  [50/81] 3s


  [60/81] 4s


  [70/81] 4s


  [80/81] 5s

Grid search selesai dalam 5s

=== Top 10 kandidat dgn sample TRAIN >= 30 trade ===
 total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct signal_mode  bb_entry_pct  rsi_entry  sl_atr_mult  max_hold
          128         35.16           1.32   149.05            -34.57        both          0.05       25.0          1.5        12
          129         33.33           1.27   125.32            -37.09         rsi          0.10       25.0          1.5        12
          129         33.33           1.27   125.32            -37.09         rsi          0.05       25.0          1.5        12
          129         33.33           1.27   125.32            -37.09        both          0.10       25.0          1.5        12
          129         33.33           1.27   125.32            -37.09         rsi          0.15       25.0          1.5        12
          129         33.33           1.27   125.32            -37.09        both          0.15       25.0          1.5    

## 6. Validasi TEST out-of-sample (HANYA kalau ada kandidat TRAIN yang layak)

Sesuai kriteria kejujuran yang ditetapkan di awal: validasi TEST cuma dilakukan kalau ADA
kandidat TRAIN dgn profit_factor > 1.5 DAN sample >= 30 trade. Kalau tidak ada, kesimpulannya
berhenti di sini -- tidak fishing for significance dgn maksa lanjut ke TEST.

In [9]:
candidates_passing = grid_valid[grid_valid["profit_factor"] > 1.5]
print(f"Kandidat TRAIN dgn PF > 1.5 DAN sample >= {MIN_SAMPLE_TRAIN}: {len(candidates_passing)}")

if len(candidates_passing) == 0:
    print("\n>>> TIDAK ADA kandidat yang lolos kriteria TRAIN. Validasi TEST DIBATALKAN (sesuai kriteria kejujuran).")
else:
    best_params = {k: candidates_passing.iloc[0][k] for k in GRID.keys()}
    print(f"\nKandidat terbaik TRAIN: {best_params}")
    print("TRAIN:", candidates_passing.iloc[0][["total_trades", "win_rate_pct", "profit_factor", "net_pnl", "max_drawdown_pct"]].to_dict())

    trades_test = run_backtest_mean_reversion(df_test, **best_params)
    test_metrics = evaluate(trades_test, INITIAL_EQUITY)
    print("\nTEST (out-of-sample):", test_metrics)

    if test_metrics["total_trades"] < MIN_SAMPLE_TEST:
        print(f"\n>>> PERINGATAN: sample TEST ({test_metrics['total_trades']}) < minimum ({MIN_SAMPLE_TEST}) -- hasil TIDAK bisa dipercaya penuh.")
    elif test_metrics["profit_factor"] > 1.5:
        print("\n>>> LOLOS validasi TEST out-of-sample -- kandidat layak dipertimbangkan lebih lanjut.")
    else:
        print("\n>>> GAGAL validasi TEST (PF turun di bawah 1.5 out-of-sample) -- kemungkinan overfitting ke TRAIN.")

Kandidat TRAIN dgn PF > 1.5 DAN sample >= 30: 0

>>> TIDAK ADA kandidat yang lolos kriteria TRAIN. Validasi TEST DIBATALKAN (sesuai kriteria kejujuran).


## 7. Kesimpulan

*(diisi setelah lihat hasil Section 5-6 -- placeholder, JANGAN diisi sblm eksekusi selesai)*